In [ ]:
# ============================================================
# 1) DOWNLOAD Swissdox -> df_raw  |  CLEAN -> df_articles
# ============================================================

import os, time, re, html
from io import BytesIO
from datetime import datetime

import requests
import pandas as pd
import yaml
from dotenv import load_dotenv

# --- ENV / KEYS ---
load_dotenv()
API_KEY = os.getenv("SWISSDOX_API_KEY")
API_SECRET = os.getenv("SWISSDOX_API_SECRET")
if not API_KEY or not API_SECRET:
    raise RuntimeError("Swissdox API keys missing. Set SWISSDOX_API_KEY and SWISSDOX_API_SECRET in .env")

API_BASE_URL   = "https://swissdox.linguistik.uzh.ch/api"
API_URL_QUERY  = f"{API_BASE_URL}/query"
API_URL_STATUS = f"{API_BASE_URL}/status"

# ✅ Conformément à la doc: seulement les headers d’auth
HEADERS = {
    "X-API-Key": API_KEY,
    "X-API-Secret": API_SECRET,
}

# --- PARAMS ---
QUERY_BASE_NAME = "BuerokratieVerwaltung_2025"
QUERY_NAME = f"{QUERY_BASE_NAME}_{datetime.now():%Y%m%d_%H%M%S}"
QUERY_COMMENT   = "Requête générée depuis VS Code"
EXPIRATION_DATE = "2026-01-30"

START_DATE = "2025-01-01"
END_DATE   = "2025-12-31"
LANGUAGES  = ["de", "fr"]
SOURCES    = ["NZZO","NNTA","NNHEU","ZWSO","TPS","NZZ","TA","ZWAO","TPSO","HEU","ZWAS","NZZS","ZWAI"]
MAX_RESULTS = 20000

# --- CLEANING ---
def clean_text(s: str) -> str:
    if not isinstance(s, str): return ""
    s = html.unescape(s).replace("\r"," ").replace("\n"," ").replace("\t"," ")
    s = re.sub(r"\s+", " ", s).strip()
    return s.strip(' "“”„\'')

def clean_xml_swissdox(s: str) -> str:
    if not isinstance(s, str): return ""
    s = html.unescape(s)
    s = re.sub(r"</p>", "\n", s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# --- YAML QUERY ---
# --- Listes (plus lisible / maintenable) ---
DE_TERMS = [
    "Bürokratie",
    "Berner Verwaltung",
    "Papierkrieg",
    "Verwaltung",
    "Bundesverwaltung",
    "Beamtenapparat",
    "Amtsschimmel",
    "Regulierungsdichte",
    "Behörden",
    "Bürokraten",
    "Beamte",
    "Staatsangestellte",
]

DE_LEVEL = ["Bund", "Bundes", "Kanton", "kantonal", "Schweiz"]

FR_TERMS = [
    "Bureaucratie",
    "Administration publique",
    "Administration fédérale",
    "Appareil administratif",
    "Appareil étatique",
    "Appareil de l’État",
    "Autorités administratives",
    "Services de l’État",
    "Services publics",
    "Fonction publique",
    "Pouvoir administratif",
    "Autorités cantonales",
    "Administration centrale",
    "Départements fédéraux",
    "Offices fédéraux",
    "Organes de l’État",
    "Technocratie",
    "Bureaucrates",
    "Fonctionnaires",
    "Employés de l'État",
]

FR_LEVEL = ["fédéral", "federal", "federale", "cantonal", "cantonale", "Suisse"]

DEPARTMENTS = [
    # DDPS / VBS
    "VBS",
    "DDPS",
    "Eidgenössische Departement für Verteidigung, Bevölkerungsschutz und Sport",
    "Département fédéral de la défense, de la protection de la population et des sports",

    # DFAE / EDA
    "EDA",
    "DFAE",
    "Eidgenössische Departement für auswärtige Angelegenheiten",
    "Département fédéral des affaires étrangères",

    # DETEC / UVEK
    "UVEK",
    "DETEC",
    "Eidgenössische Departement für Umwelt, Verkehr, Energie und Kommunikation",
    "Département fédéral de l'environnement, des transports, de l'énergie et de la communication",

    # EJPD / DFJP
    "EJPD",
    "DFJP",
    "Eidgenössische Justiz- und Polizeidepartement",
    "Département fédéral de justice et police",

    # DFI / EDI 
    "EDI",
    "DFI",
    "Eidgenössische Departement des Innern",
    "Département fédéral de l'intérieur",

    # DFF / EFD 
    "EFD",
    "DFF",
    "Eidgenössische Finanzdepartement",
    "Département fédéral des finances",

    # DEFR / WBF
    "WBF",
    "DEFR",
    "Eidgenössische Departement für Wirtschaft, Bildung und Forschung",
    "Département fédéral de l'économie, de la formation et de la recherche",
]

# --- Query block ---
query_block = {
    "sources": SOURCES,
    "dates": [{"from": START_DATE, "to": END_DATE}],
    "languages": LANGUAGES,
    "content": {
        "OR": [
            # (DE terms) AND (DE federal/cantonal level)
            {"AND": [{"OR": DE_TERMS}, {"OR": DE_LEVEL}]},

            # (FR terms) AND (FR federal/cantonal level)
            {"AND": [{"OR": FR_TERMS}, {"OR": FR_LEVEL}]},

            # Departments / acronyms / full names (no extra constraint)
            {"OR": DEPARTMENTS},
        ]
    }
}


yaml_payload = {
    "query": query_block,
    "result": {
        "format": "TSV",
        "maxResults": MAX_RESULTS,
        "columns": [
            "id","pubtime","medium_code","medium_name","rubric","regional",
            "doctype","doctype_description","language","char_count","dateline",
            "head","subhead","content_id","content",
        ],
    },
    "version": "1.2",
}

yaml_query = yaml.safe_dump(yaml_payload, sort_keys=False, allow_unicode=True)

# --- SUBMIT /query ---
payload = {
    "query": yaml_query,
    "name": QUERY_NAME,
    "comment": QUERY_COMMENT,
    "expirationDate": EXPIRATION_DATE,
    # Astuce debug: tu peux décommenter pour tester la validité du YAML sans lancer un gros job
    # "test": "1",
}

r = requests.post(API_URL_QUERY, headers=HEADERS, data=payload)

# ✅ debug si erreur (super utile pour voir le message serveur)
if r.status_code >= 400:
    print("❌ Swissdox /query failed")
    print("status:", r.status_code)
    print("response headers:", dict(r.headers))
    print("response body (first 2000 chars):")
    print(r.text[:2000])
    print("\nSent headers:", dict(r.request.headers))
    body = r.request.body
    if isinstance(body, (bytes, bytearray)):
        body = body[:800].decode("utf-8", errors="replace")
    else:
        body = str(body)[:800]
    print("\nSent body (first 800 chars):")
    print(body)

r.raise_for_status()

resp_json = r.json()
if resp_json.get("result") != "ok":
    raise SystemExit(f"❌ Swissdox non-ok: {resp_json}")

query_id = resp_json.get("queryId") or resp_json.get("id")
if not query_id:
    raise SystemExit(f"❌ queryId introuvable: {resp_json}")
print(f"✅ queryId = {query_id}")

# --- POLL /status until downloadUrl ---
download_url = None
for _ in range(300):
    rs = requests.get(API_URL_STATUS, headers=HEADERS)
    rs.raise_for_status()
    status_list = rs.json()
    job_info = next((j for j in status_list if j.get("id") == query_id), None)

    if not job_info:
        time.sleep(5); continue

    if job_info.get("error"):
        raise SystemExit(f"❌ Swissdox error: {job_info['error']}")

    download_url = job_info.get("downloadUrl")
    if download_url:
        break

    time.sleep(5)

if not download_url:
    raise SystemExit("❌ downloadUrl manquant (pas de fichier).")

# --- DOWNLOAD ---
if download_url.startswith("http"):
    download_full_url = download_url
elif download_url.startswith("/"):
    download_full_url = f"{API_BASE_URL}{download_url}"
else:
    download_full_url = f"{API_BASE_URL}/download/{download_url}"

print("⬇️ Download:", download_full_url)
r_dl = requests.get(download_full_url, headers=HEADERS)
r_dl.raise_for_status()

df_raw = pd.read_csv(BytesIO(r_dl.content), sep="\t", compression="xz")
print("✅ df_raw:", df_raw.shape)

# --- CLEAN DF -> df_articles ---
df_articles = df_raw.copy()

if "pubtime" in df_articles.columns:
    df_articles["pubtime"] = pd.to_datetime(df_articles["pubtime"].astype(str), errors="coerce", utc=True).dt.date

TEXT_COLS = ["medium_name","rubric","dateline","head","subhead"]
for c in TEXT_COLS:
    if c in df_articles.columns:
        df_articles[c] = df_articles[c].apply(clean_text)

if "content" in df_articles.columns:
    df_articles["content"] = df_articles["content"].apply(clean_xml_swissdox).apply(clean_text)

print("✅ df_articles (clean):", df_articles.shape)

In [3]:
# ============================================================
# 2) THEMES via embeddings + cosine -> df_articles_themed
# ============================================================

from sentence_transformers import SentenceTransformer
import numpy as np
import torch
import pandas as pd

# --- texte à encoder (head + subhead, fallback content) ---
CSV_PATH = "df_articles.csv"
df_articles = pd.read_csv(CSV_PATH)
df_articles_themed = df_articles.copy()

df_articles_themed["text_for_theme"] = (
    df_articles_themed["head"].fillna("").astype(str).str.strip()
    + " — "
    + df_articles_themed["subhead"].fillna("").astype(str).str.strip()
).str.strip(" —")

if "content" in df_articles_themed.columns:
    mask_empty = df_articles_themed["text_for_theme"].eq("")
    df_articles_themed.loc[mask_empty, "text_for_theme"] = (
        df_articles_themed.loc[mask_empty, "content"].fillna("").astype(str).str.slice(0, 1200)
    )

# --- taxonomy ---
THEMES = [
    (
        "Foreign Affairs",
        "FR: diplomatie, relations internationales, commerce extérieur, coopération, ambassade, consulat, ONU/UE. "
        "DE: Diplomatie, Aussenbeziehungen, Aussenhandel, Zusammenarbeit, Botschaft, Konsulat, UNO/EU."
    ),
    (
        "Culture",
        "FR: culture, art, musique, théâtre, musée, patrimoine. "
        "DE: Kultur, Kunst, Musik, Theater, Museum, Kulturerbe."
    ),
    (
        "Health",
        "FR: santé, médecins, hôpitaux, soins, LAMal, primes, pandémie. "
        "DE: Gesundheit, Ärzte, Spitäler, Pflege, KVG, Prämien, Pandemie."
    ),
    (
        "Social",
        "FR: affaires sociales, personnes âgées, AVS/AI, pauvreté, aides sociales. "
        "DE: Sozialwesen, Senioren, AHV/IV, Armut, Sozialhilfe."
    ),
    (
        "Justice",
        "FR: droit, tribunaux, police, criminalité, justice, surveillance. "
        "DE: Recht, Gerichte, Polizei, Kriminalität, Justiz, Überwachung."
    ),
    (
        "Migration",
        "FR: asile, migration, immigration, réfugiés, permis, étrangers, SEM. "
        "DE: Asyl, Migration, Einwanderung, Flüchtlinge, Bewilligungen, Ausländer, SEM."
    ),
    (
        "Defence",
        "FR: défense, armée, sécurité, protection civile, cyberattaque. "
        "DE: Verteidigung, Armee, Sicherheit, Bevölkerungsschutz, Cyberangriff."
    ),
    (
        "Sport",
        "FR: sport, clubs, fédérations, compétitions, promotion du sport. "
        "DE: Sport, Vereine, Verbände, Wettkämpfe, Sportförderung."
    ),
    (
        "Finance",
        "FR: finances publiques, budget, impôts, TVA, fiscalité. "
        "DE: Finanzen, Budget, Steuern, MWST, Fiskalpolitik."
    ),
    (
        "Economy",
        "FR: économie, entreprises, banques, commerce, marché du travail. "
        "DE: Wirtschaft, Unternehmen, Banken, Handel, Arbeitsmarkt."
    ),
    (
        "Education",
        "FR: école, université, collège, formation, enseignants, élèves. "
        "DE: Schule, Universität, Gymnasium, Bildung, Lehrpersonen, Schüler."
    ),
    (
        "Research",
        "FR: recherche, innovation, science, laboratoires, technologie. "
        "DE: Forschung, Innovation, Wissenschaft, Labore, Technologie."
    ),
    (
        "Environment",
        "FR: écologie, climat, CO2, biodiversité, protection de la nature. "
        "DE: Umwelt, Klima, CO2, Biodiversität, Naturschutz."
    ),
    (
        "Transports",
        "FR: transports, routes, mobilité, CFF, trains, voitures, avions. "
        "DE: Verkehr, Strassen, Mobilität, SBB, Züge, Autos, Flugzeuge."
    ),
    (
        "Energy",
        "FR: énergie, nucléaire, électricité, gaz, pétrole, charbon. "
        "DE: Energie, Atomkraft, Strom, Gas, Öl, Kohle."
    ),
    (
        "Communication",
        "FR: communication, médias, TV, radio, internet, réseaux, antennes. "
        "DE: Kommunikation, Medien, Fernsehen, Radio, Internet, Netze, Antennen."
    )
]


theme_labels = [t[0] for t in THEMES]
theme_texts  = [f"{t[0]}: {t[1]}" for t in THEMES]

# --- device (Mac: MPS si dispo, sinon CPU) ---
device = "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
st_model = SentenceTransformer(MODEL_NAME, device=device)

theme_emb = st_model.encode(theme_texts, normalize_embeddings=True, show_progress_bar=False)
texts = df_articles_themed["text_for_theme"].fillna("").astype(str).tolist()

emb_articles = st_model.encode(
    texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

scores = emb_articles @ theme_emb.T
best_idx = scores.argmax(axis=1)
best_score = scores.max(axis=1)

df_articles_themed["main_theme"]  = [theme_labels[i] for i in best_idx]
df_articles_themed["theme_score"] = best_score.astype(float)

THRESHOLD = 0.25
df_articles_themed.loc[df_articles_themed["theme_score"] < THRESHOLD, "main_theme"] = "Others"

print("✅ df_articles_themed:", df_articles_themed.shape)
print(df_articles_themed["main_theme"].value_counts().head(20))


Batches:   0%|          | 0/138 [00:00<?, ?it/s]

✅ df_articles_themed: (8784, 18)
main_theme
Others             3220
Defence            1214
Finance             768
Foreign Affairs     500
Migration           398
Environment         365
Justice             328
Economy             294
Culture             267
Social              248
Transports          240
Health              235
Research            230
Education           192
Communication       188
Energy               56
Sport                41
Name: count, dtype: int64


In [5]:
# ============================================================
# 3) SENTENCE SPLIT + keyword filter -> df_sentences
# ============================================================

import numpy as np
import pandas as pd
import re

df_sentences_source = df_articles_themed.copy()

if "content" not in df_sentences_source.columns:
    raise RuntimeError("Colonne 'content' absente.")

# mots-clés (phrases)
KW_SENT = ["Bürokratie",
    "Berner Verwaltung",
    "Papierkrieg",
    "Verwaltung",
    "Bundesverwaltung",
    "Beamtenapparat",
    "Amtsschimmel",
    "Regulierungsdichte",
    "Behörden",
    "Bürokraten",
    "Beamte",
    "Staatsangestellte",
    "Bureaucratie",
    "Administration publique",
    "Administration fédérale",
    "Appareil administratif",
    "Appareil étatique",
    "Appareil de l’État",
    "Autorités administratives",
    "Services de l’État",
    "Services publics",
    "Fonction publique",
    "Pouvoir administratif",
    "Autorités cantonales",
    "Administration centrale",
    "Départements fédéraux",
    "Offices fédéraux",
    "Organes de l’État",
    "Technocratie",
    "Bureaucrates",
    "Fonctionnaires",
    "Employés de l'État",
    "VBS",
    "DDPS",
    "Eidgenössische Departement für Verteidigung, Bevölkerungsschutz und Sport",
    "Département fédéral de la défense, de la protection de la population et des sports",
    "EDA",
    "DFAE",
    "Eidgenössische Departement für auswärtige Angelegenheiten",
    "Département fédéral des affaires étrangères",
    "UVEK",
    "DETEC",
    "Eidgenössische Departement für Umwelt, Verkehr, Energie und Kommunikation",
    "Département fédéral de l'environnement, des transports, de l'énergie et de la communication",
    "EJPD",
    "DFJP",
    "Eidgenössische Justiz- und Polizeidepartement",
    "Département fédéral de justice et police",
    "EDI",
    "DFI",
    "Eidgenössische Departement des Innern",
    "Département fédéral de l'intérieur",
    "EFD",
    "DFF",
    "Eidgenössische Finanzdepartement",
    "Département fédéral des finances",
    "WBF",
    "DEFR",
    "Eidgenössische Departement für Wirtschaft, Bildung und Forschung",
    "Département fédéral de l'économie, de la formation et de la recherche",]

# regex "contient un des mots-clés"
def build_kw_pattern(keywords):
    patterns = []
    for k in keywords:
        if k.isupper() and len(k) <= 4:   # acronyms like EDI, EDA, EFD
            patterns.append(rf"\b{re.escape(k)}\b")
        else:
            patterns.append(re.escape(k))
    return re.compile("|".join(patterns), flags=re.IGNORECASE)

kw_pattern = build_kw_pattern(KW_SENT)


# 1) split -> explode
tmp = df_sentences_source.copy()
tmp["sentence"] = tmp["content"].fillna("").astype(str)

# split sur . ! ? (heuristique simple)
tmp["sentence"] = tmp["sentence"].str.split(r"(?<=[.!?])\s+", regex=True)
tmp = tmp.explode("sentence", ignore_index=True)
tmp["sentence"] = tmp["sentence"].astype(str).str.strip()
tmp = tmp[tmp["sentence"].ne("")]

# 2) filtrer les phrases contenant un keyword
mask = tmp["sentence"].str.contains(kw_pattern, na=False)
tmp = tmp[mask].copy()

# 3) matched keywords (unique, join)
tmp["matched_keywords"] = tmp["sentence"].str.findall(kw_pattern).apply(
    lambda lst: ", ".join(sorted(set([x.strip() for x in lst if isinstance(x, str)])))
)

# 4) sentence_id + garder l’index article source si utile
tmp.insert(0, "sentence_id", range(1, len(tmp) + 1))
tmp["article_row_index"] = tmp.get("article_row_index", np.nan)  # optionnel si tu en avais besoin avant

# 5) df_sentences = phrases + TOUTES colonnes article propagées
# 5) df_sentences = phrases + colonnes choisies
colonnes_a_garder = [
    "sentence_id",
    "id",
    "pubtime",
    "medium_name",
    "language",
    "matched_keywords",
    "sentence",
    "main_theme",
    "theme_score",
]

df_sentences = tmp[colonnes_a_garder].copy()

print("✅ df_sentences:", df_sentences.shape)
print(df_sentences.head(5))

✅ df_sentences: (22969, 9)
    sentence_id        id     pubtime        medium_name language  \
6             1  55866222  2025-01-21  20 minuten online       de   
19            2  55866222  2025-01-21  20 minuten online       de   
25            3  58405477  2025-09-30          24 heures       fr   
26            4  58405477  2025-09-30          24 heures       fr   
33            5  58308460  2025-09-21  20 minuten online       de   

           matched_keywords  \
6                  Behörden   
19                 Behörden   
25           fonctionnaires   
26  Administration fédérale   
33                      EDA   

                                             sentence main_theme  theme_score  
6   Seit Monaten werden Websites von Schweizer Ban...    Defence     0.335271  
19  Sie nutzt vor allem DDoS-Attacken, um Websites...    Defence     0.335271  
25  Le département du conseiller fédéral Albert Rö...     Others     0.231062  
26  Cette mesure, qui s’inscrit dans le cadre d’un.

In [9]:
# ============================================================
# 4) SENTIMENT per sentence -> df_final (phrases + theme + sentiment)
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from dotenv import load_dotenv
import os

SENT_COL = "sentence"
if SENT_COL not in df_sentences.columns:
    raise RuntimeError(f"Colonne '{SENT_COL}' absente dans df_sentences.")

# --- HF token (optionnel) ---
load_dotenv()
HF_TOKEN = os.getenv("HUGGINGFACE_HUB_TOKEN")

model_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
tokenizer_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
model_kwargs     = {"token": HF_TOKEN} if HF_TOKEN else {}

tokenizer = AutoTokenizer.from_pretrained(model_name, **tokenizer_kwargs)
model = AutoModelForSequenceClassification.from_pretrained(model_name, **model_kwargs)

# device: cuda / mps / cpu
if torch.cuda.is_available():
    pipe_device = 0
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    pipe_device = "mps"
else:
    pipe_device = -1

sent_pipe = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=pipe_device)

texts = df_sentences[SENT_COL].fillna("").astype(str).tolist()

batch_size = 64
labels, scores = [], []

for i in range(0, len(texts), batch_size):
    preds = sent_pipe(texts[i:i+batch_size], truncation=True, max_length=256)
    for p in preds:
        labels.append(p["label"])
        scores.append(float(p["score"]))

df_final = df_sentences.copy()
df_final["sentiment_label"] = labels
df_final["sentiment_score"] = scores

# mapping fréquent LABEL_0/1/2 -> neg/neu/pos (si besoin)
label_map = {"LABEL_0":"negative", "LABEL_1":"neutral", "LABEL_2":"positive"}
df_final["sentiment_label"] = df_final["sentiment_label"].map(lambda x: label_map.get(x, x))

print("✅ df_final:", df_final.shape)
print(df_final[["sentence_id","main_theme","theme_score","sentiment_label","sentiment_score","matched_keywords","sentence"]].head(5))

# --- export ---
df_final.to_csv("Swissdox_phrases_theme_sentiment.csv", index=False, encoding="utf-8-sig")


Device set to use mps


✅ df_final: (22969, 11)
    sentence_id main_theme  theme_score sentiment_label  sentiment_score  \
6             1    Defence     0.335271         neutral         0.760287   
19            2    Defence     0.335271         neutral         0.683726   
25            3     Others     0.231062        positive         0.619053   
26            4     Others     0.231062        positive         0.750350   
33            5     Others     0.234409         neutral         0.787656   

           matched_keywords                                           sentence  
6                  Behörden  Seit Monaten werden Websites von Schweizer Ban...  
19                 Behörden  Sie nutzt vor allem DDoS-Attacken, um Websites...  
25           fonctionnaires  Le département du conseiller fédéral Albert Rö...  
26  Administration fédérale  Cette mesure, qui s’inscrit dans le cadre d’un...  
33                      EDA  Ein EDA-Gutachten zur Anerkennung Palästinas b...  
